# RG3 저장 모델 예측 (재학습 없음)


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import nbformat
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'rg3/01_preprocess_train.ipynb').exists())
n=nbformat.read(ROOT/'rg3/01_preprocess_train.ipynb',4)
exec(n.cells[1].source) # 정의만 실행. 학습 cell은 실행하지 않음.
INPUT_CSV=ROOT/'dataset/processed/rg3/rg3_input_features.csv'
f=pd.read_csv(INPUT_CSV,parse_dates=['TimeStamp']).drop(columns=['PassOrFail','Reason'],errors='ignore')
gas_bundle=load_bundle(MODEL_DIR/'gas12.joblib')
short_bundle=joblib.load(MODEL_DIR/'shortshot_best.joblib') # 이 저장소에서 직접 생성한 신뢰 모델만 로드
out=predict_gas_features(gas_bundle,f)
short,score=predict_short(short_bundle,f)
out['shortshot_candidate']=short.astype(int)
out['any_defect_candidate']=(out.gas_prediction.eq(1)|short).astype(int)
out['predicted_type']=np.select([out.status.ne('ok'),out.gas_prediction.eq(1)&short,out.gas_prediction.eq(1),short],['판정보류','가스+미성형 후보','가스 후보','미성형 후보'],default='경보 없음(정상 확정 아님)')
# 두 모델은 모두 실행하므로 하나가 양성이라고 다른 모델을 건너뛰지 않는다.
outdir=ROOT/'rg3/predictions';outdir.mkdir(exist_ok=True)
out.to_csv(outdir/'rg3_predictions.csv',index=False,encoding='utf-8-sig')
print('FULLFIT inference, training samples are not validation:',len(out))
print(out.predicted_type.value_counts())
display(out.head())

